# E10 — Which process-state features improve cross-product battery disassembly ranking?

This is the next focused experiment after E9. It keeps E9's operation features,
dimensions, MLP capacity, training settings, candidate convention, and state updater.
It changes **which state-feature families update after the current operation**.
Unused families retain their initial values; no columns are dropped or zeroed.

| Variant | Global state fractions | Component activity and completion | Local degrees and fixation types |
|---|---|---|---|
| Static | Frozen | Frozen | Frozen |
| GlobalOnly | Updated | Frozen | Frozen |
| GlobalStatus | Updated | Updated | Frozen |
| GlobalStructure | Updated | Frozen | Updated |
| Full | Updated | Updated | Updated |

Each variant is evaluated with MLP and XGBRanker: ten conditions × ten products ×
three seeds = **300 fit records**. With successful verification of all 120 saved E9
static/full anchors, only **180 new fits** are needed. If any anchor check fails,
all 120 anchors are trained again, giving 300 new fits. The notebook never reruns E9.

**Colab workflow**
1. Upload this notebook using **File → Upload notebook** and choose a GPU runtime.
2. Keep the existing E9 result folder on Drive, or upload the E9 result ZIP in the
   Files panel. Set `E9_RESULTS_PATH` in Cell 2 if it is not found automatically.
3. Run Cells 1–10. They lock the protocol, check features, run reduced diagnostic
   fits, and verify saved anchors. These diagnostic scores are not study results.
4. Check the reported new-fit budget. The full run is disabled by default.
   When ready, set `RUN_FULL_EXPERIMENT=True` **in Cell 11** and run that cell.
5. Run Cells 12–13 after completion and share `e10_state_feature_families_results.zip`.

Every completed fit is saved to Drive. After disconnection, rerun setup/preflight,
then Cell 11: validated E10 fits are skipped. Only an interrupted fit restarts.
No previous E9 result is overwritten. All five variants retain normalized sequence
progress and initial engineering attributes; GlobalOnly means updating the three
additional state fractions, not removing those common inputs.

The follow-up design was selected after seeing E9. Its protocol is written before
any E10 training; this is not a claim that E10 was planned before E9. Results measure
teacher-forced reference-order ranking, not physical feasibility or optimal plans.
E11 adjacency controls remain a separate experiment and are not implemented here.

## Before the full run — Establish the precise literature gap

The candidate contribution is the **joint audit of engineering state, physical
edge alignment, and transfer to unseen products**. The individual ingredients
below already exist. This short comparison identifies close work; it does not
establish a first-study claim or complete the novelty review.

| Close work | Established contribution to account for | Comparison needed in the revised paper |
|---|---|---|
| [Yin et al., 2025](https://doi.org/10.1016/j.jclepro.2025.145558) | Knowledge-graph embeddings and GIN for battery disassembly sequencing | Compare task, product holdout, process-state inputs, and control design using the full paper |
| [Xiao et al., 2022](https://doi.org/10.1016/j.cirpj.2022.07.010) | Dynamic Bayesian disassembly representation and sequence optimization | Explain why updating a disassembly representation is not the novelty |
| [Katsman et al., 2026](https://doi.org/10.1007/s11063-026-11833-6) | Strong MLP comparisons and features that subsume structural information | Distinguish the process-dependent engineering audit from a generic MLP-versus-GNN result |
| [ASAP](https://asap.csail.mit.edu/) | Learned GNN part-selection priorities within physically constrained assembly/disassembly search | Keep reference-order prediction distinct from physical planning and compare representation evaluation |

Expand this comparison against the closest full papers before committing to the
full compute budget. If equivalent joint controls already exist, further runs alone
will not resolve the novelty problem. E10 asks which supplied state summaries add
predictive value; it cannot prove physical causality or general GNN inferiority.

## Cell 1 — Dependencies and imports
Install missing packages without replacing Colab PyTorch or restarting the runtime.

In [ ]:
import sys, importlib.util, subprocess
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'torch': 'torch',
          'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.2',
          'xgboost': 'xgboost>=2,<4', 'matplotlib': 'matplotlib', 'packaging': 'packaging',
          'IPython': 'ipython'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import os, gc, io, json, math, random, time, tarfile, zipfile
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import hashlib, itertools, shutil, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.preprocessing import OneHotEncoder
from packaging.version import Version
import torch
import torch.nn as nn
import torch.nn.functional as F
import xgboost as xgb
import matplotlib.pyplot as plt
from IPython.display import display
if Version(xgb.__version__) < Version('2.0') or Version(sklearn.__version__) < Version('1.2'):
    raise RuntimeError('Install xgboost>=2 and scikit-learn>=1.2 in a clean Colab runtime.')
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
print({'python': sys.version.split()[0], 'torch': torch.__version__,
       'xgboost': xgb.__version__, 'sklearn': sklearn.__version__})

## Cell 2 — Fixed design and durable output directory
Keep the scientific settings unchanged for the planned experiment. Drive is reused
if already mounted. An explicit E9 directory should contain `protocol.json`, `fits`,
`models`, and `diagnostics`; a ZIP may contain those folders at any one nested level.
A missing/incompatible source means fresh anchors, not invented or prefilled results.

In [ ]:
SEEDS = [17, 29, 43]
EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32
BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119
DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE
XGB_TREES = 250
CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_DRIVE = True
SAVE_MODELS = True
REUSE_E9_ANCHORS = True
AUTO_DOWNLOAD_DATASET = True
DATA_PATH = ''
E9_RESULTS_PATH = ''  # E9 result ZIP, exact result folder, or one-run parent folder.
MAX_NEW_FITS_THIS_CALL = None  # Optional positive integer; rerun Cell 11 to continue.
LOCAL_OUTPUT_ROOT = '/content/ev_e10' if IN_COLAB else './ev_e10'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/EV_E10_State_Feature_Families'
LOCAL_CACHE_ROOT = Path('/content/ev_e10_cache' if IN_COLAB else './ev_e10_cache').resolve()
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
EXPECTED_E9_PROTOCOL = json.loads("{\"experiment\": \"E9 state-aware non-graph baselines\", \"feature_version\": \"E9-state-scalars-v1.0\", \"implementation_sha256\": \"881566fa41124575a6d5ed0f99cb08b518548b4746e32e87e34249164c4f4483\", \"source_e6_sha256\": \"90c01dc01b2eccfb9e592de3ba649c7017a52086d7ee23260db992cb47d48d81\", \"dataset_sha256\": {\"battery_catalogue.csv\": \"a1de7eadff569c511f8aeabf1fa7013bc249f4ad450a7f9bf6a09ca1710371b3\", \"parts.csv\": \"3ceaba9d186dcc7342499e3ccf432a07c78b75dbd4cc732c0b43a92aa140c790\", \"fixations.csv\": \"5bc454a59509275b199828fc7376d28507046bffd28c3f4ec23f076d8bd52339\", \"operations.csv\": \"8cfad668c5b87765dc4de044d72c8b8ce439d67740506a422e4597dc4fc4ba05\"}, \"seeds\": [17, 29, 43], \"epochs\": 30, \"negative_samples\": 8, \"learning_rate\": 0.002, \"weight_decay\": 0.0001, \"dropout\": 0.1, \"xgb_trees\": 250, \"conditions\": [\"StaticFeatureMLP\", \"StateAwareMLP\", \"StaticFeatureXGB\", \"StateAwareXGB\", \"DynamicRealGNN\"], \"folds\": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], \"torch_version\": \"2.11.0+cu130\", \"xgboost_version\": \"3.4.1\", \"sklearn_version\": \"1.6.1\", \"numpy_version\": \"2.1.3\", \"deterministic_torch\": true, \"tf32_matmul\": false, \"evaluation_batching\": \"Canonical internal batching; shuffled candidate metrics; E6 tie tolerance 1e-8\", \"independent_unit\": \"held-out battery product\", \"state_rule\": \"Exact E6 post-current update; all original fixation classes retained\", \"mlp_capacity\": \"Width chosen by parameter-count gap only; shared by static/state models\", \"xgb_budget\": \"E3 250-tree settings; all nonsingleton training candidates, CPU hist\", \"notes\": \"No ID/order input, no test tuning; historic E6 is descriptive only\"}")
SOURCE_E9_NOTEBOOK_SHA256 = '023773bb4f03f9de5ebc221578a5aefdd3c2c5697cee048678773b18d82855b2'
IMPLEMENTATION_SHA256 = '3b6740ffdc8bcb1d388607dd0c42264954929fbbae74ccfd3e391b1e1a2ca84c'
FEATURE_VERSION = 'E10-frozen-family-v1.0'
METRICS = ['hits1', 'hits3', 'mrr']
ROLE_NAMES = ['source', 'target', 'part']
VARIANTS = ['Static', 'GlobalOnly', 'GlobalStatus', 'GlobalStructure', 'Full']
LEARNERS = ['MLP', 'XGB']
CONDITIONS = [variant + learner for learner in LEARNERS for variant in VARIANTS]
PRIMARY_PAIRS = [('Full' + learner, variant + learner) for learner in LEARNERS
                 for variant in ['GlobalOnly', 'GlobalStatus', 'GlobalStructure']]
PREFLIGHT_PASSED = False
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive is not mounted. Retry, or explicitly set USE_DRIVE=False.')
    OUTPUT_ROOT = Path(DRIVE_OUTPUT_ROOT)
else:
    OUTPUT_ROOT = Path(LOCAL_OUTPUT_ROOT).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert MAX_NEW_FITS_THIS_CALL is None or (isinstance(MAX_NEW_FITS_THIS_CALL, int) and MAX_NEW_FITS_THIS_CALL > 0)
print('MLP device:', DEVICE, '| XGB: CPU histogram, threads:', CPU_THREADS)
print('Output directory:', OUTPUT_ROOT)
print('E10 conditions:', CONDITIONS)

## Cell 3 — Original KIT data
Reuse local data or download the same public dataset as E9. CSV hashes enter the locked protocol.

In [ ]:
CSV_NAMES = ['battery_catalogue.csv', 'parts.csv', 'fixations.csv', 'operations.csv']
DATASET_URL = 'https://radar.kit.edu/radar-backend/archives/emz24pksshndq468/versions/1/content'
search_root = Path('/content') if IN_COLAB else Path.cwd()
def search_local_csvs(root):
    for folder, directories, filenames in os.walk(root):
        directories[:] = [d for d in directories if d not in {'drive', '.git', 'node_modules', '__pycache__'}]
        if set(CSV_NAMES).issubset(filenames):
            return Path(folder)
    return None
def extract_dataset_archive(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:*') as tf:
        resolved = destination.resolve()
        for member in tf.getmembers():
            target = (resolved/member.name).resolve()
            if not target.is_relative_to(resolved) or member.issym() or member.islnk():
                raise ValueError('Unsafe archive entry: '+member.name)
        if sys.version_info >= (3, 12):
            tf.extractall(destination, filter='data')
        else:
            tf.extractall(destination)
    result = search_local_csvs(destination)
    if result is None:
        raise FileNotFoundError('Four dataset CSVs were not found in the archive.')
    return result
if DATA_PATH:
    supplied = Path(DATA_PATH).expanduser()
    if supplied.is_file():
        DATA = extract_dataset_archive(supplied, search_root/'ev_kit_dataset')
    elif supplied.is_dir():
        DATA = search_local_csvs(supplied)
    else:
        raise FileNotFoundError(DATA_PATH)
else:
    DATA = search_local_csvs(search_root)
if DATA is None:
    archives = list(search_root.glob('10.35097-*.tar'))
    if archives:
        DATA = extract_dataset_archive(archives[0], search_root/'ev_kit_dataset')
    elif AUTO_DOWNLOAD_DATASET:
        archive = search_root/'10.35097-emz24pksshndq468.tar'
        if not archive.exists():
            print('Downloading the public KIT dataset ...')
            temp = archive.with_suffix('.tar.part')
            request = urllib.request.Request(DATASET_URL, headers={'User-Agent': 'EV-research-notebook/1.0'})
            with urllib.request.urlopen(request, timeout=90) as response, temp.open('wb') as stream:
                shutil.copyfileobj(response, stream)
            os.replace(temp, archive)
        DATA = extract_dataset_archive(archive, search_root/'ev_kit_dataset')
    else:
        raise FileNotFoundError('Upload the KIT .tar or set DATA_PATH to its extracted CSV directory.')
if DATA is None:
    raise FileNotFoundError('The configured folder does not contain the four KIT CSVs.')
DATASET_HASHES = {name: hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in CSV_NAMES}
bat = pd.read_csv(DATA/'battery_catalogue.csv')
parts = pd.read_csv(DATA/'parts.csv')
fix = pd.read_csv(DATA/'fixations.csv')
ops = pd.read_csv(DATA/'operations.csv')
# Preserve the column convention used by E6; accept case variations from a CSV export.
rename = {c: {'frompartid': 'fromPartID', 'topartid': 'toPartID'}.get(c.lower(), c) for c in fix.columns}
fix = fix.rename(columns=rename)
actual_counts = (len(bat), len(parts), len(fix), len(ops))
assert actual_counts == (10, 563, 1896, 2281), f'Unexpected dataset version/counts: {actual_counts}'
for frame, required in [(bat, ['id', 'name']), (parts, ['id', 'class', 'battery_id']),
                        (fix, ['id', 'class', 'battery_id', 'fromPartID', 'toPartID']),
                        (ops, ['id', 'skill_id', 'fixation_id', 'part_id', 'next_op_id'])]:
    assert set(required).issubset(frame.columns), set(required)-set(frame.columns)
    assert frame.id.is_unique and frame.id.notna().all()
assert (ops.fixation_id.notna() ^ ops.part_id.notna()).all()
print('Data:', DATA)
print('Counts (products, parts, fixations, operations):', actual_counts)
display(bat[['id', 'name']])

## Cell 4 — Original E9 chain and training-only vocabulary
Raw IDs reconstruct references and serve as lookup keys; they do not enter model features.

In [ ]:
battery_names = bat.set_index('id')['name'].to_dict()
fix_battery = fix.set_index('id')['battery_id'].to_dict()
part_battery = parts.set_index('id')['battery_id'].to_dict()
part_class_raw = parts.set_index('id')['class'].to_dict()
fix_info_raw = fix.set_index('id')[['class', 'fromPartID', 'toPartID']].to_dict('index')

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])
ops['battery_id'] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index('id')
chains = {}
for b in sorted(ops['battery_id'].unique()):
    sub = ops[ops['battery_id'] == b]
    ids = set(sub['id'].astype(int))
    next_ids = set(sub['next_op_id'].dropna().astype(int))
    starts = list(ids - next_ids)
    terminals = sub[sub['next_op_id'].isna()]['id'].astype(int).tolist()
    assert len(starts) == 1
    assert len(terminals) == 1
    seq = []
    seen = set()
    cur = int(starts[0])
    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)
        nxt = ops_by_id.loc[cur, 'next_op_id']
        if pd.isna(nxt):
            break
        cur = int(nxt)
    assert len(seq) == len(ids), f'Broken chain for battery {b}'
    chains[int(b)] = seq
print({battery_names[b]: len(seq) for b, seq in chains.items()})

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)
    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi['fromPartID'])])
        dst_cls = str(part_class_raw[int(fi['toPartID'])])
        return f"{skill}|fix:{fi['class']}|{src_cls}->{dst_cls}"
    return f'{skill}|part:{part_class_raw[int(r.part_id)]}'
raw_signatures = {b: [raw_full_signature(op_id) for op_id in seq] for b, seq in chains.items()}

def make_vocab(values):
    vals = sorted({str(x) for x in values if pd.notna(x)})
    return {'<UNK>': 0, **{v: i + 1 for i, v in enumerate(vals)}}

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {'part': make_vocab(parts[parts['battery_id'].isin(train_batteries)]['class']), 'fix': make_vocab(fix[fix['battery_id'].isin(train_batteries)]['class']), 'skill': make_vocab(ops[ops['battery_id'].isin(train_batteries)]['skill_id'])}

def build_battery_pack(battery_id, vocab, graph_mode='real', shuffle_id=0):
    assert graph_mode in {'real', 'shuffle', 'noedge'}
    psub = parts[parts['battery_id'] == battery_id].copy()
    fsub = fix[fix['battery_id'] == battery_id].copy()
    part_id_to_local = {int(pid): i for i, pid in enumerate(psub['id'].astype(int))}
    n_nodes = len(psub)
    true_degree = np.zeros(n_nodes, dtype=np.float32)
    edge_src = []
    edge_dst = []
    edge_type = []
    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]
        true_degree[a] += 1
        true_degree[b] += 1
        edge_src += [a, b]
        edge_dst += [b, a]
        et = lookup(vocab['fix'], r['class'])
        edge_type += [et, et]
    true_degree_norm = true_degree / max(1.0, float(true_degree.max()))
    node_part_class = np.array([lookup(vocab['part'], cls) for cls in psub['class']], dtype=np.int64)
    true_edge_index = np.array([edge_src, edge_dst], dtype=np.int64)
    true_edge_type = np.array(edge_type, dtype=np.int64)
    if graph_mode == 'real':
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()
    elif graph_mode == 'shuffle':
        rng_graph = np.random.default_rng(GRAPH_SHUFFLE_BASE + 10007 * int(shuffle_id) + int(battery_id))
        permutation = rng_graph.permutation(n_nodes)
        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()
        shuffled_degree = np.bincount(graph_edge_index.reshape(-1), minlength=n_nodes).astype(np.float32)
        graph_degree_norm = shuffled_degree / max(1.0, float(shuffled_degree.max()))
    else:
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)
        graph_degree_norm = np.zeros(n_nodes, dtype=np.float32)
    seq = [int(x) for x in chains[battery_id]]
    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []
    for op_id in seq:
        r = ops_by_id.loc[op_id]
        skill_idx = lookup(vocab['skill'], r.skill_id)
        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]
            src_pid = int(fi['fromPartID'])
            dst_pid = int(fi['toPartID'])
            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]
            categorical.append([skill_idx, 0, lookup(vocab['fix'], fi['class']), lookup(vocab['part'], part_class_raw[src_pid]), lookup(vocab['part'], part_class_raw[dst_pid]), 0])
            graph_nodes.append([src_node, dst_node, -1])
            numeric.append([true_degree_norm[src_node], true_degree_norm[dst_node], 0.0])
        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]
            categorical.append([skill_idx, 1, 0, 0, 0, lookup(vocab['part'], part_class_raw[pid])])
            graph_nodes.append([-1, -1, node])
            numeric.append([0.0, 0.0, true_degree_norm[node]])
        signatures.append(raw_full_signature(op_id))
    return {'battery_id': int(battery_id), 'battery_name': battery_names[int(battery_id)], 'node_part_class': node_part_class, 'node_degree': graph_degree_norm.astype(np.float32), 'edge_index': graph_edge_index, 'edge_type': graph_edge_type, 'op_categorical': np.asarray(categorical, dtype=np.int64), 'op_graph_nodes': np.asarray(graph_nodes, dtype=np.int64), 'op_numeric': np.asarray(numeric, dtype=np.float32), 'full_signatures': signatures, 'n_ops': len(seq), 'graph_mode': graph_mode, 'shuffle_id': int(shuffle_id)}

In [ ]:
FOLD_IDS = sorted(chains)
assert len(FOLD_IDS) == 10 and sum(len(seq)-1 for seq in chains.values()) == 2271
print('Planned full fits:', len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS))

## Cell 5 — Unchanged neural architecture and tie-aware metrics
The GNN class is retained only to calculate the original capacity target; E10 trains no GNN.

In [ ]:
class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(self, n_skill, n_fix, n_part, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)
        op_input_dim = emb_dim + 4 + emb_dim + emb_dim + emb_dim + emb_dim + 3
        self.op_mlp = nn.Sequential(nn.Linear(op_input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.score_mlp = nn.Sequential(nn.Linear(hidden_dim * 4 + 1, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(pack['op_categorical'], dtype=torch.long, device=DEVICE)
        nums = torch.as_tensor(pack['op_numeric'], dtype=torch.float32, device=DEVICE)
        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]
        x = torch.cat([self.skill_emb(skill), self.kind_emb(kind), self.fix_emb(fix_class), self.part_emb(src_class), self.part_emb(dst_class), self.part_emb(part_class), nums], dim=-1)
        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([current_repr, candidate_repr, torch.abs(current_repr - candidate_repr), current_repr * candidate_repr, progress], dim=-1)
        return self.score_mlp(x).squeeze(-1)

class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(self, n_part, n_fix, part_emb_dim=EMBED_DIM, edge_emb_dim=8, graph_dim=GNN_DIM, dropout=DROPOUT):
        super().__init__()
        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)
        self.input_proj = nn.Linear(part_emb_dim + 1, graph_dim)
        self.message_layers = nn.ModuleList([nn.Linear(graph_dim + edge_emb_dim, graph_dim) for _ in range(2)])
        self.self_layers = nn.ModuleList([nn.Linear(graph_dim, graph_dim) for _ in range(2)])
        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(pack['node_part_class'], dtype=torch.long, device=DEVICE)
        degree = torch.as_tensor(pack['node_degree'], dtype=torch.float32, device=DEVICE).unsqueeze(-1)
        h = F.relu(self.input_proj(torch.cat([self.part_emb(node_class), degree], dim=-1)))
        edge_index = torch.as_tensor(pack['edge_index'], dtype=torch.long, device=DEVICE)
        edge_type = torch.as_tensor(pack['edge_type'], dtype=torch.long, device=DEVICE)
        src = edge_index[0]
        dst = edge_index[1]
        edge_embedding = self.edge_emb(edge_type)
        for message_layer, self_layer in zip(self.message_layers, self.self_layers):
            msg = message_layer(torch.cat([h[src], edge_embedding], dim=-1))
            agg = torch.zeros((h.shape[0], h.shape[1]), dtype=h.dtype, device=DEVICE)
            agg.index_add_(0, dst, msg)
            counts = torch.zeros((h.shape[0], 1), dtype=h.dtype, device=DEVICE)
            counts.index_add_(0, dst, torch.ones((dst.numel(), 1), dtype=h.dtype, device=DEVICE))
            agg = agg / counts.clamp_min(1.0)
            h = F.relu(self_layer(h) + agg)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

class GNNOperationRanker(BaseOperationRanker):

    def __init__(self, n_skill, n_fix, n_part, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, graph_dim=GNN_DIM, dropout=DROPOUT):
        super().__init__(n_skill=n_skill, n_fix=n_fix, n_part=n_part, emb_dim=emb_dim, hidden_dim=hidden_dim, dropout=dropout)
        self.graph_encoder = GraphEncoder(n_part=n_part, n_fix=n_fix, part_emb_dim=emb_dim, graph_dim=graph_dim, dropout=dropout)
        self.structural_projection = nn.Sequential(nn.Linear(graph_dim * 3, hidden_dim), nn.ReLU())
        self.combine = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU())

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)
        op_nodes = torch.as_tensor(pack['op_graph_nodes'], dtype=torch.long, device=DEVICE)
        n_ops = op_nodes.shape[0]
        a = torch.zeros((n_ops, node_h.shape[1]), device=DEVICE)
        b = torch.zeros_like(a)
        fixation_mask = op_nodes[:, 0] >= 0
        if fixation_mask.any():
            a[fixation_mask] = node_h[op_nodes[fixation_mask, 0]]
            b[fixation_mask] = node_h[op_nodes[fixation_mask, 1]]
        part_mask = op_nodes[:, 2] >= 0
        if part_mask.any():
            a[part_mask] = node_h[op_nodes[part_mask, 2]]
        structural = self.structural_projection(torch.cat([a, b, torch.abs(a - b)], dim=-1))
        return self.combine(torch.cat([base, structural], dim=-1))

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = ([], [], [], [])
    for t in range(n_ops - 2):
        pool = np.arange(t + 2, n_ops, dtype=np.int64)
        neg = rng.choice(pool, size=k, replace=len(pool) < k)
        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(t / max(1, n_ops - 1))
    return (np.asarray(current, dtype=np.int64), np.asarray(positive, dtype=np.int64), np.stack(negative), np.asarray(progress, dtype=np.float32))

In [ ]:
def expected_tie_metrics(scores, target_position=0, tol=1e-08):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """
    scores = np.asarray(scores)
    target_score = scores[target_position]
    greater = int(np.sum(scores > target_score + tol))
    equal = int(np.sum(np.abs(scores - target_score) <= tol))
    equal = max(equal, 1)
    hits1 = max(0, min(equal, 1 - greater)) / equal if greater < 1 else 0.0
    hits3 = max(0, min(equal, 3 - greater)) / equal if greater < 3 else 0.0
    ranks = np.arange(greater + 1, greater + equal + 1, dtype=float)
    mrr = float(np.mean(1.0 / ranks))
    return (hits1, hits3, mrr)

def count_trainable_parameters(model):
    return sum((p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def training_signature_set(train_batteries):
    s = set()
    for b in train_batteries:
        s.update(raw_signatures[b])
    return s

## Cell 6 — Unchanged E6 state updater
Retain fixation records and the original simplified handling of fixed connections.
A component can remain structurally active after completion under that convention.
Changing the updater here would introduce another intervention into E10.

In [ ]:
from collections import defaultdict
DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE

def _battery_structure_records(battery_id, vocab):
    psub = parts[parts['battery_id'] == battery_id].copy()
    fsub = fix[fix['battery_id'] == battery_id].copy()
    part_ids = psub['id'].astype(int).tolist()
    part_id_to_local = {pid: i for i, pid in enumerate(part_ids)}
    records = []
    for _, r in fsub.iterrows():
        records.append({'fixation_id': int(r['id']), 'u': part_id_to_local[int(r.fromPartID)], 'v': part_id_to_local[int(r.toPartID)], 'edge_type': lookup(vocab['fix'], r['class'])})
    return (psub, part_id_to_local, records)

def _dynamic_permutation(battery_id, n_nodes, shuffle_id):
    rng = np.random.default_rng(DYNAMIC_SHUFFLE_BASE + 10007 * int(shuffle_id) + int(battery_id))
    return rng.permutation(n_nodes)

def _state_edges_and_degree(n_nodes, records, removed_fixations, active_nodes, initial_max_degree):
    edge_src = []
    edge_dst = []
    edge_type = []
    degree = np.zeros(n_nodes, dtype=np.float32)
    for rec in records:
        if rec['fixation_id'] in removed_fixations:
            continue
        u = int(rec['u'])
        v = int(rec['v'])
        if not active_nodes[u] or not active_nodes[v]:
            continue
        degree[u] += 1.0
        degree[v] += 1.0
        edge_src += [u, v]
        edge_dst += [v, u]
        edge_type += [int(rec['edge_type']), int(rec['edge_type'])]
    if len(edge_src):
        edge_index = np.asarray([edge_src, edge_dst], dtype=np.int64)
        edge_type = np.asarray(edge_type, dtype=np.int64)
    else:
        edge_index = np.empty((2, 0), dtype=np.int64)
        edge_type = np.empty((0,), dtype=np.int64)
    degree_norm = (degree / max(1.0, float(initial_max_degree))).astype(np.float32)
    return (edge_index, edge_type, degree_norm)

def _complete_state_edges_and_degree(active_nodes):
    active = np.flatnonzero(active_nodes).astype(np.int64)
    n_nodes = len(active_nodes)
    if len(active) <= 1:
        return (np.empty((2, 0), dtype=np.int64), np.empty((0,), dtype=np.int64), np.zeros(n_nodes, dtype=np.float32))
    src = np.repeat(active, len(active))
    dst = np.tile(active, len(active))
    keep = src != dst
    src = src[keep]
    dst = dst[keep]
    edge_index = np.asarray([src, dst], dtype=np.int64)
    edge_type = np.zeros(len(src), dtype=np.int64)
    degree_norm = np.zeros(n_nodes, dtype=np.float32)
    degree_norm[active] = (len(active) - 1) / max(1, n_nodes - 1)
    return (edge_index, edge_type, degree_norm)

def _has_remaining_incident_fixation(node, records, removed_fixations, active_nodes):
    """
    Causal check: does this currently active node still participate in any
    modeled fixation that has not yet been executed and whose other endpoint
    is also structurally active?
    """
    for rec in records:
        if rec['fixation_id'] in removed_fixations:
            continue
        u = int(rec['u'])
        v = int(rec['v'])
        if not active_nodes[u] or not active_nodes[v]:
            continue
        if node == u or node == v:
            return True
    return False

def _deactivate_completed_parts(records, removed_fixations, active_nodes, part_completed):
    """
    Deactivate a part only after:
      1) its part operation has already occurred, and
      2) it has no remaining active incident fixation.

    Iterate to a fixed point because deactivating one completed isolated node
    can make another completed node isolated as well.
    """
    changed = True
    while changed:
        changed = False
        pending = np.flatnonzero(part_completed & active_nodes)
        for node in pending:
            node = int(node)
            if not _has_remaining_incident_fixation(node=node, records=records, removed_fixations=removed_fixations, active_nodes=active_nodes):
                active_nodes[node] = False
                changed = True
    return active_nodes

def build_e6_state_pack(battery_id, vocab, condition, shuffle_id=0):
    """
    Conditions:
      static_real
      dynamic_real
      dynamic_noedge
      dynamic_complete
      dynamic_shuffle

    Returns the invariant operation pack plus one block-diagonal graph
    containing all post-current ranking states t=0,...,n_ops-2.
    """
    assert condition in {'static_real', 'dynamic_real', 'dynamic_noedge', 'dynamic_complete', 'dynamic_shuffle'}
    base_pack = build_battery_pack(battery_id, vocab, graph_mode='real', shuffle_id=0)
    psub, part_id_to_local, true_records = _battery_structure_records(battery_id, vocab)
    n_nodes = len(psub)
    n_states = base_pack['n_ops'] - 1
    initial_degree = np.zeros(n_nodes, dtype=np.float32)
    for rec in true_records:
        initial_degree[rec['u']] += 1.0
        initial_degree[rec['v']] += 1.0
    initial_max_degree = max(1.0, float(initial_degree.max()))
    if condition == 'dynamic_shuffle':
        permutation = _dynamic_permutation(battery_id, n_nodes, shuffle_id)
        records = [{'fixation_id': rec['fixation_id'], 'u': int(permutation[rec['u']]), 'v': int(permutation[rec['v']]), 'edge_type': rec['edge_type']} for rec in true_records]
        event_node_map = permutation
    else:
        permutation = np.arange(n_nodes, dtype=np.int64)
        records = [dict(rec) for rec in true_records]
        event_node_map = permutation
    static_edge_index, static_edge_type, static_degree = _state_edges_and_degree(n_nodes=n_nodes, records=records, removed_fixations=set(), active_nodes=np.ones(n_nodes, dtype=bool), initial_max_degree=initial_max_degree)
    removed_fixations = set()
    active_nodes = np.ones(n_nodes, dtype=bool)
    part_completed = np.zeros(n_nodes, dtype=bool)
    state_edge_indices = []
    state_edge_types = []
    state_degrees = []
    state_active_fraction = []
    state_edge_fraction = []
    initial_undirected_edges = len(true_records)
    seq = [int(x) for x in chains[battery_id]]
    for t in range(n_states):
        op_id = seq[t]
        r = ops_by_id.loc[op_id]
        if condition != 'static_real':
            if pd.notna(r.fixation_id):
                removed_fixations.add(int(r.fixation_id))
            else:
                semantic_node = part_id_to_local[int(r.part_id)]
                structural_node = int(event_node_map[semantic_node])
                part_completed[structural_node] = True
            active_nodes = _deactivate_completed_parts(records=records, removed_fixations=removed_fixations, active_nodes=active_nodes, part_completed=part_completed)
        if condition == 'static_real':
            edge_index = static_edge_index
            edge_type = static_edge_type
            degree_norm = static_degree
            active_fraction = 1.0
            edge_fraction = 1.0
        elif condition == 'dynamic_noedge':
            edge_index = np.empty((2, 0), dtype=np.int64)
            edge_type = np.empty((0,), dtype=np.int64)
            degree_norm = np.zeros(n_nodes, dtype=np.float32)
            active_fraction = float(active_nodes.mean())
            edge_fraction = 0.0
        elif condition == 'dynamic_complete':
            edge_index, edge_type, degree_norm = _complete_state_edges_and_degree(active_nodes)
            active_fraction = float(active_nodes.mean())
            max_directed = n_nodes * max(0, n_nodes - 1)
            edge_fraction = edge_index.shape[1] / max(1, max_directed)
        else:
            edge_index, edge_type, degree_norm = _state_edges_and_degree(n_nodes=n_nodes, records=records, removed_fixations=removed_fixations, active_nodes=active_nodes, initial_max_degree=initial_max_degree)
            active_fraction = float(active_nodes.mean())
            edge_fraction = edge_index.shape[1] / 2.0 / max(1, initial_undirected_edges)
        state_edge_indices.append(edge_index)
        state_edge_types.append(edge_type)
        state_degrees.append(degree_norm)
        state_active_fraction.append(active_fraction)
        state_edge_fraction.append(edge_fraction)
    block_src = []
    block_dst = []
    block_type = []
    for state_idx, (edge_index, edge_type) in enumerate(zip(state_edge_indices, state_edge_types)):
        offset = state_idx * n_nodes
        if edge_index.shape[1]:
            block_src.append(edge_index[0] + offset)
            block_dst.append(edge_index[1] + offset)
            block_type.append(edge_type)
    if block_src:
        block_edge_index = np.asarray([np.concatenate(block_src), np.concatenate(block_dst)], dtype=np.int64)
        block_edge_type = np.concatenate(block_type).astype(np.int64)
    else:
        block_edge_index = np.empty((2, 0), dtype=np.int64)
        block_edge_type = np.empty((0,), dtype=np.int64)
    block_pack = {'node_part_class': np.tile(base_pack['node_part_class'], n_states).astype(np.int64), 'node_degree': np.concatenate(state_degrees).astype(np.float32), 'edge_index': block_edge_index, 'edge_type': block_edge_type}
    return {'base_pack': base_pack, 'state_graph_pack': block_pack, 'n_nodes': int(n_nodes), 'n_states': int(n_states), 'state_active_fraction': np.asarray(state_active_fraction, dtype=np.float32), 'state_edge_fraction': np.asarray(state_edge_fraction, dtype=np.float32), 'condition': condition, 'shuffle_id': int(shuffle_id), 'permutation': permutation.astype(np.int64)}

def build_e6_model(vocab):
    return GNNOperationRanker(n_skill=len(vocab['skill']), n_fix=len(vocab['fix']), n_part=len(vocab['part'])).to(DEVICE)

def compose_state_operation_repr(model, base_repr, node_h, base_pack, n_nodes, state_indices, op_indices):
    state_indices = torch.as_tensor(state_indices, dtype=torch.long, device=DEVICE)
    op_indices = torch.as_tensor(op_indices, dtype=torch.long, device=DEVICE)
    op_nodes_all = torch.as_tensor(base_pack['op_graph_nodes'], dtype=torch.long, device=DEVICE)
    op_nodes = op_nodes_all[op_indices]
    offsets = state_indices * int(n_nodes)
    n_items = len(op_indices)
    a = torch.zeros((n_items, node_h.shape[1]), dtype=node_h.dtype, device=DEVICE)
    b = torch.zeros_like(a)
    fixation_mask = op_nodes[:, 0] >= 0
    if fixation_mask.any():
        a[fixation_mask] = node_h[offsets[fixation_mask] + op_nodes[fixation_mask, 0]]
        b[fixation_mask] = node_h[offsets[fixation_mask] + op_nodes[fixation_mask, 1]]
    part_mask = op_nodes[:, 2] >= 0
    if part_mask.any():
        a[part_mask] = node_h[offsets[part_mask] + op_nodes[part_mask, 2]]
    structural = model.structural_projection(torch.cat([a, b, torch.abs(a - b)], dim=-1))
    return model.combine(torch.cat([base_repr[op_indices], structural], dim=-1))

## Cell 7 — Original E9 summaries and capacity-matched MLP
All feature columns and operation descriptors are preserved.

In [ ]:
def build_scalar_state_pack(battery_id, vocab):
    """Reproduce E6's post-current state without exposing adjacency to a learner."""
    base = build_battery_pack(battery_id, vocab, graph_mode="real")
    _, part_map, records = _battery_structure_records(battery_id, vocab)
    n = len(part_map)
    n_states = base["n_ops"] - 1
    n_types = len(vocab["fix"])
    roles = base["op_graph_nodes"]
    valid = roles >= 0
    safe_roles = np.maximum(roles, 0)
    initial = np.zeros(n, dtype=np.float32)
    for r in records:
        initial[r["u"]] += 1
        initial[r["v"]] += 1
    denom = max(1.0, float(initial.max()))
    removed = set()
    active = np.ones(n, dtype=bool)
    complete = np.zeros(n, dtype=bool)
    local_names = []
    for group in ["degree_norm", "degree_fraction", "active", "completed"]:
        local_names.extend(f"{group}_{role}" for role in ROLE_NAMES)
    type_labels = [k for k, _ in sorted(vocab["fix"].items(), key=lambda kv: kv[1])]
    for role in ROLE_NAMES:
        local_names.extend(f"incident_type_{role}_{label}" for label in type_labels)

    def snapshot():
        degree = np.zeros(n, dtype=np.float32)
        typed = np.zeros((n, n_types), dtype=np.float32)
        retained = 0
        for r in records:
            if r["fixation_id"] in removed or not (active[r["u"]] and active[r["v"]]):
                continue
            retained += 1
            for node in [r["u"], r["v"]]:
                degree[node] += 1
                typed[node, r["edge_type"]] += 1
        group = [degree / denom, degree / np.maximum(initial, 1.0),
                 active.astype(np.float32), complete.astype(np.float32)]
        local = [x[safe_roles] * valid for x in group]
        incident = typed[safe_roles] * valid[..., None] / denom
        local.append(incident.reshape(base["n_ops"], -1))
        local = np.concatenate(local, axis=1).astype(np.float32)
        global_features = np.array([retained / max(1, len(records)),
                                    active.mean(), complete.mean()], dtype=np.float32)
        return local, global_features, degree / denom

    initial_local, initial_global, _ = snapshot()
    local_states, global_states, node_degrees = [], [], []
    for t in range(n_states):
        row = ops_by_id.loc[int(chains[battery_id][t])]
        if pd.notna(row.fixation_id):
            removed.add(int(row.fixation_id))
        else:
            complete[part_map[int(row.part_id)]] = True
        _deactivate_completed_parts(records, removed, active, complete)
        local, global_features, degree = snapshot()
        local_states.append(local)
        global_states.append(global_features)
        node_degrees.append(degree)

    # Raw indices are used here only to calculate identity-overlap summaries.
    # They never become a model feature or a vocabulary category.
    overlaps = np.zeros((base["n_ops"], base["n_ops"], 3), dtype=np.float32)
    touched = [set(int(x) for x in row if x >= 0) for row in roles]
    for i, a in enumerate(touched):
        for j, b in enumerate(touched):
            shared = len(a & b)
            overlaps[i, j] = [float(shared > 0), shared / 2.0,
                              shared / max(1, len(a | b))]
    return {"base_pack": base, "local": np.stack(local_states),
            "global": np.stack(global_states), "initial_local": initial_local,
            "initial_global": initial_global, "overlaps": overlaps,
            "node_degrees": np.stack(node_degrees), "local_names": local_names,
            "n_nodes": n}

def scalar_inputs(item, states, operations, state_aware):
    states = np.asarray(states, dtype=np.int64)
    operations = np.asarray(operations, dtype=np.int64)
    if state_aware:
        return item["local"][states, operations]
    return item["initial_local"][operations]

def pair_extra(item, states, current, candidate, state_aware):
    states = np.asarray(states, dtype=np.int64)
    if state_aware:
        global_values = item["global"][states]
    else:
        global_values = np.repeat(item["initial_global"][None], len(states), axis=0)
    return np.concatenate([global_values, item["overlaps"][current, candidate]], axis=1)

class ScalarOperationRanker(BaseOperationRanker):
    """Shared operation MLP plus local state scalars; no graph/message passing."""
    def __init__(self, vocab, n_local, width):
        super().__init__(len(vocab["skill"]), len(vocab["fix"]), len(vocab["part"]),
                         hidden_dim=width)
        self.width = width
        self.op_mlp = nn.Sequential(nn.Linear(67 + n_local, width), nn.ReLU(),
                                    nn.Dropout(DROPOUT), nn.Linear(width, width), nn.ReLU())
        self.score_mlp = nn.Sequential(nn.Linear(width * 4 + 1 + 6, width), nn.ReLU(),
                                       nn.Dropout(DROPOUT), nn.Linear(width, 1))

    def encode_selected(self, item, states, operations, state_aware):
        base = item["base_pack"]
        operations = np.asarray(operations, dtype=np.int64)
        cats = torch.as_tensor(base["op_categorical"][operations], dtype=torch.long, device=DEVICE)
        nums = np.concatenate([base["op_numeric"][operations],
                               scalar_inputs(item, states, operations, state_aware)], axis=1)
        nums = torch.as_tensor(nums, dtype=torch.float32, device=DEVICE)
        values = [self.skill_emb(cats[:, 0]), self.kind_emb(cats[:, 1]),
                  self.fix_emb(cats[:, 2]), self.part_emb(cats[:, 3]),
                  self.part_emb(cats[:, 4]), self.part_emb(cats[:, 5]), nums]
        return self.op_mlp(torch.cat(values, dim=-1))

    def score_selected(self, item, states, current, candidate, state_aware,
                       current_repr=None, candidate_repr=None):
        cur = current_repr if current_repr is not None else self.encode_selected(item, states, current, state_aware)
        cand = candidate_repr if candidate_repr is not None else self.encode_selected(item, states, candidate, state_aware)
        progress = np.asarray(states, dtype=np.float32) / max(1, item["base_pack"]["n_ops"] - 1)
        extra = np.column_stack([progress, pair_extra(item, states, current, candidate, state_aware)])
        extra = torch.as_tensor(extra, dtype=torch.float32, device=DEVICE)
        return self.score_mlp(torch.cat([cur, cand, torch.abs(cur-cand), cur*cand, extra], dim=-1)).squeeze(-1)

def matched_mlp_width(vocab, n_local):
    """Analytical parameter matching; does not inspect any validation/test score."""
    target = count_trainable_parameters(GNNOperationRanker(
        len(vocab["skill"]), len(vocab["fix"]), len(vocab["part"])))
    # Three embedding tables plus the 2-by-4 kind table.
    embeddings = 12 * (len(vocab["skill"]) + len(vocab["fix"]) + len(vocab["part"])) + 8
    counts = {w: embeddings + 5*w*w + (67+n_local+11)*w + 1 for w in range(32, 161)}
    width = min(counts, key=lambda w: (abs(counts[w] - target), w))
    actual = count_trainable_parameters(ScalarOperationRanker(vocab, n_local, width))
    assert actual == counts[width], (actual, counts[width])
    return width, actual, target

## Cell 8 — Unchanged MLP and XGB training and evaluation
Tree and neural training budgets remain distinct. Stable scoring batches preserve the original near-tie convention.

In [ ]:
def train_scalar_mlp(items, train_batteries, vocab, seed, state_aware, epochs=EPOCHS):
    set_seed(seed)
    n_local = items[train_batteries[0]]["local"].shape[-1]
    width, params, target = matched_mlp_width(vocab, n_local)
    # Width search must not consume the model initialization RNG.
    set_seed(seed)
    model = ScalarOperationRanker(vocab, n_local, width).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    losses = []
    for epoch in range(epochs):
        order = list(train_batteries)
        rng.shuffle(order)
        model.train()
        epoch_losses = []
        for b in order:
            item = items[b]
            cur, pos, neg, _ = make_training_pairs(item["base_pack"]["n_ops"], NEGATIVES_PER_STATE, rng)
            k = neg.shape[1]
            current_repr = model.encode_selected(item, cur, cur, state_aware)
            pos_score = model.score_selected(item, cur, cur, pos, state_aware, current_repr=current_repr)
            repeated_current = current_repr[:, None].expand(-1, k, -1).reshape(-1, model.width)
            neg_score = model.score_selected(item, np.repeat(cur, k), np.repeat(cur, k),
                                              neg.reshape(-1), state_aware,
                                              current_repr=repeated_current).reshape(len(cur), k)
            loss = F.softplus(-(pos_score[:, None] - neg_score)).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss for battery {b}, epoch {epoch+1}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        losses.append({"epoch": epoch+1, "loss": float(np.mean(epoch_losses))})
    return model, {"width": width, "parameter_count": params, "gnn_parameter_count": target,
                   "capacity_gap_fraction": (params-target)/target, "loss_curve": losses}

def make_scalar_encoder(vocab):
    op = [np.arange(len(vocab["skill"])), np.arange(2), np.arange(len(vocab["fix"]))]
    op += [np.arange(len(vocab["part"])) for _ in range(3)]
    encoder = OneHotEncoder(categories=op+op, handle_unknown="ignore",
                            sparse_output=True, dtype=np.float32)
    encoder.fit(np.zeros((1, 12), dtype=np.int64))
    return encoder

def scalar_pair_matrix(item, t, candidates, encoder, state_aware):
    candidates = np.asarray(candidates, dtype=np.int64)
    base = item["base_pack"]
    n = len(candidates)
    cur = np.full(n, t, dtype=np.int64)
    cur_cat = base["op_categorical"][cur]
    cand_cat = base["op_categorical"][candidates]
    cat = encoder.transform(np.column_stack([cur_cat, cand_cat]))
    cur_local = np.column_stack([base["op_numeric"][cur], scalar_inputs(item, cur, cur, state_aware)])
    cand_local = np.column_stack([base["op_numeric"][candidates], scalar_inputs(item, cur, candidates, state_aware)])
    numeric = np.column_stack([cur_local, cand_local, np.abs(cur_local-cand_local),
                               (cur_cat == cand_cat).astype(np.float32),
                               np.full(n, t/max(1, base["n_ops"]-1), dtype=np.float32),
                               pair_extra(item, cur, cur, candidates, state_aware)])
    return sparse.hstack([cat, sparse.csr_matrix(numeric)], format="csr", dtype=np.float32)

def training_xgb_matrix(items, train_batteries, encoder, state_aware):
    matrices, labels, queries = [], [], []
    rng = np.random.default_rng(BASE_SEED)
    qid = 0
    for b in train_batteries:
        item = items[b]
        n_ops = item["base_pack"]["n_ops"]
        for t in range(n_ops-2):
            candidates = rng.permutation(np.arange(t+1, n_ops))
            matrices.append(scalar_pair_matrix(item, t, candidates, encoder, state_aware))
            labels.append((candidates == t+1).astype(np.float32))
            queries.append(np.full(len(candidates), qid, dtype=np.int32))
            qid += 1
    return sparse.vstack(matrices, format="csr"), np.concatenate(labels), np.concatenate(queries)

def make_scalar_xgb(seed, n_estimators=XGB_TREES):
    return xgb.XGBRanker(objective="rank:pairwise", eval_metric="ndcg@3",
                         n_estimators=n_estimators, max_depth=6, learning_rate=0.05,
                         min_child_weight=1.0, subsample=0.90, colsample_bytree=0.90,
                         reg_lambda=1.0, reg_alpha=0.0, tree_method="hist",
                         n_jobs=CPU_THREADS, device="cpu", random_state=int(seed))

def candidate_inventory(n_ops, prefix):
    # Sequence indices are internal lookup keys, never numeric model inputs.
    return np.setdiff1d(np.arange(n_ops), np.asarray(prefix), assume_unique=True)

def evaluate_scalar_or_gnn(model, item, kind, state_aware=None, encoder=None, candidate_seed=None):
    base = item["base_pack"]
    n_ops = base["n_ops"]
    rng = np.random.default_rng(BASE_SEED + 1009 * int(base["battery_id"])
                                if candidate_seed is None else candidate_seed)
    if kind in {"mlp", "gnn"}:
        model.eval()
    rows = []
    with torch.no_grad():
        if kind == "gnn":
            base_repr = model.base_operation_encoding(base)
            node_h = model.graph_encoder(item["state_graph_pack"])
        for t in range(n_ops-1):
            inventory = candidate_inventory(n_ops, range(t+1))
            candidates = rng.permutation(inventory)
            # Fixed internal batching preserves E6's floating-point tie behavior.
            # Lookup order is never a feature. Shuffle scores back before the metric.
            scoring_candidates = inventory
            states = np.full(len(candidates), t, dtype=np.int64)
            if kind == "mlp":
                scores = model.score_selected(item, states, states, scoring_candidates, state_aware).cpu().numpy()
            elif kind == "xgb":
                scores = model.predict(scalar_pair_matrix(item, t, scoring_candidates, encoder, state_aware))
            elif kind == "gnn":
                current = compose_state_operation_repr(model, base_repr, node_h, base,
                            item["n_nodes"], np.array([t]), np.array([t])).expand(len(candidates), -1)
                candidate = compose_state_operation_repr(model, base_repr, node_h, base,
                            item["n_nodes"], states, scoring_candidates)
                progress = torch.full((len(candidates), 1), t/max(1, n_ops-1), device=DEVICE)
                scores = model.score_pairs(current, candidate, progress).cpu().numpy()
            else:
                raise ValueError(kind)
            if not np.isfinite(scores).all():
                raise FloatingPointError("Non-finite evaluation scores")
            scores = scores[np.searchsorted(inventory, candidates)]
            target = int(np.flatnonzero(candidates == t+1)[0])
            h1, h3, mrr = expected_tie_metrics(scores, target_position=target)
            rows.append({"position": t, "candidate_size": len(candidates),
                         "progress": t/max(1, n_ops-1), "hits1": h1, "hits3": h3, "mrr": mrr})
    detail = pd.DataFrame(rows)
    return {m: float(detail[m].mean()) for m in METRICS}, detail

In [ ]:
def validate_scalar_states(items, vocab):
    rows = []
    for b, item in items.items():
        real = build_e6_state_pack(b, vocab, "dynamic_real")
        degree = real["state_graph_pack"]["node_degree"].reshape(real["n_states"], real["n_nodes"])
        np.testing.assert_array_equal(degree, item["node_degrees"])
        np.testing.assert_allclose(real["state_edge_fraction"], item["global"][:, 0], atol=1e-7)
        np.testing.assert_allclose(real["state_active_fraction"], item["global"][:, 1], atol=1e-7)
        assert np.isfinite(item["local"]).all()
        assert item["local"].min() >= 0 and item["local"].max() <= 1.00001
        assert np.all(np.diff(item["global"][:, 0]) <= 1e-7)
        assert np.all(np.diff(item["global"][:, 1]) <= 1e-7)
        assert np.all(np.diff(item["global"][:, 2]) >= -1e-7)
        rows.append({"battery_id": b, "states": real["n_states"],
                     "local_features": item["local"].shape[-1], "e6_state_match": True,
                     "final_active_fraction": float(item["global"][-1, 1])})
        del real
    return pd.DataFrame(rows)

def assert_candidate_and_metric_invariants(item, vocab):
    encoder = make_scalar_encoder(vocab)
    for t in sorted(set([0, max(0, item["base_pack"]["n_ops"]//2-1), item["base_pack"]["n_ops"]-2])):
        candidates = candidate_inventory(item["base_pack"]["n_ops"], range(t+1))
        order = np.random.default_rng(89+t).permutation(len(candidates))
        for mode in [False, True]:
            x = scalar_pair_matrix(item, t, candidates, encoder, mode)
            xp = scalar_pair_matrix(item, t, candidates[order], encoder, mode)
            assert (xp-x[order]).nnz == 0
        scores = np.linspace(-1, 1, len(candidates))
        target = int(np.flatnonzero(candidates == t+1)[0])
        expected = expected_tie_metrics(scores, target)
        actual = expected_tie_metrics(scores[order], int(np.flatnonzero(order == target)[0]))
        np.testing.assert_allclose(expected, actual)
    np.testing.assert_allclose(expected_tie_metrics(np.ones(4), 2), [0.25, 0.75, np.mean(1/np.arange(1, 5))])
    np.testing.assert_allclose(expected_tie_metrics(np.ones(1)), [1, 1, 1])

## Cell 9 — Feature-family selectors, protocol locking, and resume helpers
Static and Full reproduce the E9 endpoint selectors. The three intermediate
variants update only their named families. Exact checkpoint re-evaluation and
strict software-version checks are required before reusing **any** E9 anchors.
If one fails, none are imported. Source hashes and provenance remain in the output.

In [ ]:
def atomic_json(path, payload):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)

def atomic_csv(frame, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    compression = "gzip" if str(path).endswith(".gz") else None
    frame.to_csv(temp, index=False, compression=compression)
    os.replace(temp, path)

def fit_paths(run_dir, battery_id, seed, condition):
    key = f"battery{int(battery_id):02d}_seed{int(seed)}_{condition}"
    return key, run_dir/"fits"/(key+".json"), run_dir/"fits"/(key+"_states.csv.gz")

In [ ]:
VARIANTS = ['Static', 'GlobalOnly', 'GlobalStatus', 'GlobalStructure', 'Full']
LEARNERS = ['MLP', 'XGB']
CONDITION_SPEC = {v + k: (v, k.lower()) for k in LEARNERS for v in VARIANTS}
ANCHOR_MAP = {'StaticMLP': 'StaticFeatureMLP', 'FullMLP': 'StateAwareMLP',
              'StaticXGB': 'StaticFeatureXGB', 'FullXGB': 'StateAwareXGB'}
GLOBAL_NAMES = ['remaining_fixation_fraction', 'active_fraction', 'completed_fraction']

# Retain E9's endpoint selectors for independent numerical equivalence checks.
e9_scalar_inputs = scalar_inputs
e9_pair_extra = pair_extra


def family_masks(item):
    names = item['local_names']
    status = np.array([n.startswith(('active_', 'completed_')) for n in names])
    structure = np.array([n.startswith(('degree_norm_', 'degree_fraction_', 'incident_type_'))
                          for n in names])
    assert status.sum() == 6 and structure.sum() == len(names) - 6
    assert np.all(status ^ structure), 'Every local feature must belong to exactly one family.'
    return status, structure


def variant_name(value):
    if isinstance(value, (bool, np.bool_)):
        return 'Full' if value else 'Static'
    if value not in VARIANTS:
        raise ValueError('Unknown feature variant: ' + str(value))
    return value


def scalar_inputs(item, states, operations, state_aware):
    """Keep all columns; freeze unused families at their initial values."""
    variant = variant_name(state_aware)
    states = np.asarray(states, dtype=np.int64)
    operations = np.asarray(operations, dtype=np.int64)
    if variant in {'Static', 'GlobalOnly'}:
        return item['initial_local'][operations]
    if variant == 'Full':
        return item['local'][states, operations]
    status, structure = family_masks(item)
    updated = status if variant == 'GlobalStatus' else structure
    return np.where(updated, item['local'][states, operations], item['initial_local'][operations])


def pair_extra(item, states, current, candidate, state_aware):
    variant = variant_name(state_aware)
    states = np.asarray(states, dtype=np.int64)
    global_values = (np.repeat(item['initial_global'][None], len(states), axis=0)
                     if variant == 'Static' else item['global'][states])
    return np.concatenate([global_values, item['overlaps'][current, candidate]], axis=1)


def check_feature_families(item, vocab):
    status, structure = family_masks(item)
    n_states, n_ops, n_local = item['local'].shape
    # All state/operation combinations, including operations that are not candidates.
    states = np.repeat(np.arange(n_states), n_ops)
    operations = np.tile(np.arange(n_ops), n_states)
    initial = item['initial_local'][operations]
    dynamic = item['local'][states, operations]
    for variant in VARIANTS:
        actual = scalar_inputs(item, states, operations, variant)
        assert actual.shape == initial.shape and actual.dtype == initial.dtype
        updated = (np.ones(n_local, dtype=bool) if variant == 'Full' else
                   status if variant == 'GlobalStatus' else
                   structure if variant == 'GlobalStructure' else
                   np.zeros(n_local, dtype=bool))
        np.testing.assert_array_equal(actual[:, updated], dynamic[:, updated])
        np.testing.assert_array_equal(actual[:, ~updated], initial[:, ~updated])
        extra = pair_extra(item, states, operations, operations, variant)
        expected_global = (np.repeat(item['initial_global'][None], len(states), axis=0)
                           if variant == 'Static' else item['global'][states])
        np.testing.assert_array_equal(extra[:, :3], expected_global)
        np.testing.assert_array_equal(extra[:, 3:], item['overlaps'][operations, operations])
    for mode, variant in [(False, 'Static'), (True, 'Full')]:
        np.testing.assert_array_equal(scalar_inputs(item, states, operations, variant),
                                      e9_scalar_inputs(item, states, operations, mode))
        np.testing.assert_array_equal(pair_extra(item, states, operations, operations, variant),
                                      e9_pair_extra(item, states, operations, operations, mode))
    encoder = make_scalar_encoder(vocab)
    widths = set()
    for t in sorted({0, n_states // 2, n_states - 1}):
        candidates = candidate_inventory(n_ops, range(t + 1))
        order = np.random.default_rng(711 + t).permutation(len(candidates))
        for variant in VARIANTS:
            x = scalar_pair_matrix(item, t, candidates, encoder, variant)
            xp = scalar_pair_matrix(item, t, candidates[order], encoder, variant)
            assert (xp - x[order]).nnz == 0
            widths.add(x.shape[1])
        for mode, variant in [(False, 'Static'), (True, 'Full')]:
            assert (scalar_pair_matrix(item, t, candidates, encoder, mode) -
                    scalar_pair_matrix(item, t, candidates, encoder, variant)).nnz == 0
    assert len(widths) == 1
    return {'battery_id': int(item['base_pack']['battery_id']), 'states': n_states,
            'local_features': n_local, 'status_features': int(status.sum()),
            'structure_features': int(structure.sum()), 'xgb_features': widths.pop(),
            'all_families_checked': True, 'e9_endpoints_identical': True}


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def resolve_e9_source():
    """Explicit path first; otherwise use an unambiguous completed E9 run."""
    roots = []
    if E9_RESULTS_PATH:
        roots = [Path(E9_RESULTS_PATH).expanduser()]
    elif IN_COLAB:
        roots = sorted(Path('/content').glob('e9_state_aware_baselines_results*.zip'))
        if not roots:
            roots = [Path('/content/drive/MyDrive/EV_State_Aware_Baselines')]
    else:
        return None
    if len(roots) > 1:
        raise RuntimeError('Several E9 ZIPs found. Set E9_RESULTS_PATH to the intended run.')
    root = roots[0]
    if not root.exists():
        if E9_RESULTS_PATH:
            raise FileNotFoundError('E9_RESULTS_PATH does not exist: ' + str(root))
        return None
    if root.is_file():
        if not zipfile.is_zipfile(root):
            raise ValueError('E9_RESULTS_PATH must be a result ZIP or a result directory.')
        digest = sha256_file(root)
        destination = LOCAL_CACHE_ROOT / ('e9_' + digest[:16])
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(root) as z:
            base = destination.resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or ((member.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError('Unsafe ZIP entry: ' + member.filename)
            z.extractall(destination)
        root = destination
    candidates = [root] if (root / 'protocol.json').is_file() else sorted(
        p.parent for p in root.rglob('protocol.json') if (p.parent / 'e9_completion.json').is_file())
    if len(candidates) != 1:
        raise RuntimeError('Expected one completed E9 run. Set E9_RESULTS_PATH to its exact directory.')
    return candidates[0]


def source_manifest(source):
    if source is None:
        return {'available': False}
    paths = [source / 'protocol.json', source / 'e9_completion.json']
    paths += sorted((source / 'diagnostics').glob('features_fold*.json'))
    for b in FOLD_IDS:
        for seed in SEEDS:
            for old in ANCHOR_MAP.values():
                key = f'battery{b:02d}_seed{seed}_{old}'
                paths += [source / 'fits' / (key + '.json'),
                          source / 'fits' / (key + '_states.csv.gz'),
                          source / 'models' / (key + ('.ubj' if old.endswith('XGB') else '.pt'))]
    manifest = {str(p.relative_to(source)): sha256_file(p) if p.is_file() else 'MISSING' for p in paths}
    return {'available': True, 'files': manifest,
            'content_sha256': hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()}


def e10_protocol_payload():
    return {'experiment': 'E10 state feature families', 'feature_version': FEATURE_VERSION,
            'implementation_sha256': IMPLEMENTATION_SHA256,
            'source_e9_notebook_sha256': SOURCE_E9_NOTEBOOK_SHA256,
            'expected_e9_implementation_sha256': EXPECTED_E9_PROTOCOL['implementation_sha256'],
            'dataset_sha256': DATASET_HASHES, 'seeds': SEEDS, 'folds': FOLD_IDS,
            'conditions': CONDITIONS, 'variants': VARIANTS, 'learners': LEARNERS,
            'frozen_family': 'Initial-state values retained; no dropped or zeroed columns',
            'common_features': 'E9 static attributes, initial operation degrees, shared-component features, sequence progress',
            'epochs': EPOCHS, 'negative_samples': NEGATIVES_PER_STATE, 'learning_rate': LR,
            'base_seed': BASE_SEED, 'embed_dim': EMBED_DIM, 'hidden_dim': HIDDEN_DIM, 'gnn_dim': GNN_DIM,
            'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT, 'xgb_trees': XGB_TREES,
            'cpu_threads': CPU_THREADS, 'device': str(DEVICE),
            'torch_version': torch.__version__, 'xgboost_version': xgb.__version__,
            'sklearn_version': sklearn.__version__, 'numpy_version': np.__version__,
            'python_version': sys.version.split()[0],
            'deterministic_torch': torch.are_deterministic_algorithms_enabled(),
            'tf32_matmul': torch.backends.cuda.matmul.allow_tf32,
            'state_rule': EXPECTED_E9_PROTOCOL['state_rule'],
            'evaluation_batching': EXPECTED_E9_PROTOCOL['evaluation_batching'],
            'primary_metric': 'hits1', 'primary_pairs': [list(pair) for pair in PRIMARY_PAIRS],
            'holm_family': 'Six E10 Hits@1 contrasts', 'independent_unit': 'held-out battery product',
            'design_timing': 'Selected after E9; locked before E10 preflight and full training',
            'save_models': SAVE_MODELS, 'reuse_e9_anchors': REUSE_E9_ANCHORS,
            'e9_source_manifest': E9_SOURCE_MANIFEST,
            'reuse_rule': 'All 120 anchors must pass protocol, feature, checkpoint, and per-state metric checks; otherwise rerun all anchors',
            'scope': 'Teacher-forced reference next-operation ranking; no physical feasibility or causal claim'}


def prepare_e10_directory():
    payload = e10_protocol_payload()
    signature = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    run_dir = OUTPUT_ROOT / ('run_' + signature[:12])
    run_dir.mkdir(parents=True, exist_ok=True)
    for name in ['fits', 'models', 'diagnostics']:
        (run_dir / name).mkdir(exist_ok=True)
    path = run_dir / 'protocol.json'
    if path.exists():
        assert json.loads(path.read_text()) == payload, 'Protocol changed; use its new run directory.'
    else:
        atomic_json(path, payload)
    return run_dir, signature


def validate_detail(frame, record, b, seed, condition):
    n = len(chains[b]) - 1
    assert len(frame) == n and record['n_states'] == n
    np.testing.assert_array_equal(frame.position.to_numpy(), np.arange(n))
    np.testing.assert_array_equal(frame.candidate_size.to_numpy(), np.arange(n, 0, -1))
    assert (frame.battery_id == b).all() and (frame.seed == seed).all()
    assert (frame.condition == condition).all()
    values = frame[METRICS].to_numpy()
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
    np.testing.assert_allclose(frame[METRICS].mean().to_numpy(),
                               [record[m] for m in METRICS], rtol=0, atol=1e-12)


def verify_e9_anchors(source, run_dir):
    report = {'requested': REUSE_E9_ANCHORS, 'eligible': False, 'checked_fits': 0,
              'reason': 'No E9 source supplied; anchors will be trained afresh.'}
    if not REUSE_E9_ANCHORS or source is None:
        report['reason'] = 'Anchor reuse disabled.' if not REUSE_E9_ANCHORS else report['reason']
        atomic_json(run_dir / 'diagnostics' / 'anchor_verification.json', report)
        return {}, report
    verified = {}
    try:
        previous = json.loads((source / 'protocol.json').read_text())
        for key in ['experiment', 'feature_version', 'implementation_sha256', 'source_e6_sha256',
                    'dataset_sha256', 'seeds', 'epochs', 'negative_samples', 'learning_rate',
                    'weight_decay', 'dropout', 'xgb_trees', 'conditions', 'folds',
                    'deterministic_torch', 'tf32_matmul', 'evaluation_batching', 'state_rule']:
            assert previous[key] == EXPECTED_E9_PROTOCOL[key], 'Unexpected E9 protocol field: ' + key
        assert previous['dataset_sha256'] == DATASET_HASHES, 'Current dataset differs from E9.'
        current_settings = {'seeds': SEEDS, 'epochs': EPOCHS, 'negative_samples': NEGATIVES_PER_STATE,
                            'learning_rate': LR, 'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT,
                            'xgb_trees': XGB_TREES, 'folds': FOLD_IDS}
        for key, value in current_settings.items():
            assert previous[key] == value, 'Current scientific setting differs: ' + key
        for key, value in [('torch_version', torch.__version__), ('numpy_version', np.__version__),
                           ('xgboost_version', xgb.__version__), ('sklearn_version', sklearn.__version__)]:
            assert previous[key] == value, 'Version differs: ' + key + '; rerun all anchors in this environment.'
        assert BASE_SEED == 20261003 and (EMBED_DIM, HIDDEN_DIM, GNN_DIM) == (12, 64, 32)
        assert torch.are_deterministic_algorithms_enabled() and not torch.backends.cuda.matmul.allow_tf32
        completion = json.loads((source / 'e9_completion.json').read_text())
        assert completion['full_canonical_design'] and completion['all_requested_fits_complete']
        assert completion['finished_fits'] == 150
        signature = hashlib.sha256(json.dumps(previous, sort_keys=True).encode()).hexdigest()
        for b in FOLD_IDS:
            train = [x for x in sorted(chains) if x != b]
            vocab = build_fold_vocab(train)
            item = build_scalar_state_pack(b, vocab)
            width, params, target = matched_mlp_width(vocab, item['local'].shape[-1])
            feature_record = json.loads((source / 'diagnostics' / f'features_fold{b:02d}.json').read_text())
            assert feature_record['vocab'] == vocab and feature_record['local_features'] == item['local_names']
            assert feature_record['global_features'] == GLOBAL_NAMES
            assert feature_record['mlp_width'] == width and feature_record['mlp_parameters'] == params
            assert feature_record['gnn_parameters'] == target
            encoder = make_scalar_encoder(vocab)
            for condition, old_condition in ANCHOR_MAP.items():
                variant, kind = CONDITION_SPEC[condition]
                for seed in SEEDS:
                    key, path, states_path = fit_paths(source, b, seed, old_condition)
                    record = json.loads(path.read_text())
                    frame = pd.read_csv(states_path)
                    assert record['protocol_signature'] == signature
                    assert record['battery_id'] == b and record['seed'] == seed and record['condition'] == old_condition
                    validate_detail(frame, record, b, seed, old_condition)
                    if kind == 'mlp':
                        assert record['device'] == str(DEVICE), 'MLP device type differs; rerun all anchors.'
                        model_path = source / 'models' / (key + '.pt')
                        payload = torch.load(model_path, map_location=DEVICE, weights_only=True)
                        assert payload['vocab'] == vocab and payload['condition'] == old_condition
                        assert record['width'] == width and record['parameter_count'] == params
                        assert len(record['loss_curve']) == EPOCHS
                        assert np.isfinite([x['loss'] for x in record['loss_curve']]).all()
                        model = ScalarOperationRanker(vocab, item['local'].shape[-1], width).to(DEVICE)
                        model.load_state_dict(payload['state_dict'], strict=True)
                    else:
                        model_path = source / 'models' / (key + '.ubj')
                        model = make_scalar_xgb(seed)
                        model.load_model(model_path)
                        assert model.get_booster().num_boosted_rounds() == XGB_TREES
                        assert model.get_booster().num_features() == record['feature_count']
                    metrics, detail = evaluate_scalar_or_gnn(model, item, kind, variant, encoder)
                    # Exact per-state tie-aware metrics, not merely a rounded aggregate.
                    np.testing.assert_allclose(detail[METRICS].to_numpy(), frame[METRICS].to_numpy(), rtol=0, atol=1e-12)
                    np.testing.assert_allclose([metrics[m] for m in METRICS], [record[m] for m in METRICS], rtol=0, atol=1e-12)
                    verified[(b, seed, condition)] = {'record': record, 'model_path': model_path,
                        'states_path': states_path, 'record_sha256': sha256_file(path),
                        'states_sha256': sha256_file(states_path), 'model_sha256': sha256_file(model_path)}
                    report['checked_fits'] += 1
                    del model
            print(f'Anchor verification: product {b}, {report["checked_fits"]}/120 fits checked', flush=True)
            del item
            gc.collect()
        assert len(verified) == 120
        report.update(eligible=True, reason='All 120 anchors passed numerical and procedural checks.')
    except Exception as exc:
        verified = {}
        report['reason'] = type(exc).__name__ + ': ' + str(exc)
        print('E9 anchors will ALL be retrained:', report['reason'], flush=True)
    report['new_full_budget'] = 180 if report['eligible'] else 300
    atomic_json(run_dir / 'diagnostics' / 'anchor_verification.json', report)
    return verified, report


def completed_e10_fit(run_dir, b, seed, condition, signature):
    key, path, states_path = fit_paths(run_dir, b, seed, condition)
    if not path.exists() or not states_path.exists():
        return False
    record = json.loads(path.read_text())
    assert record['protocol_signature'] == signature
    assert record['battery_id'] == b and record['seed'] == seed and record['condition'] == condition
    assert record['states_sha256'] == sha256_file(states_path), 'Completed state file changed.'
    validate_detail(pd.read_csv(states_path), record, b, seed, condition)
    if SAVE_MODELS:
        model_path = run_dir / 'models' / (key + ('.ubj' if condition.endswith('XGB') else '.pt'))
        assert model_path.is_file() and record['model_sha256'] == sha256_file(model_path)
    return True


def commit_e10_fit(run_dir, signature, b, seed, condition, metrics, detail, extra,
                   model=None, reused=None, vocab=None):
    key, path, states_path = fit_paths(run_dir, b, seed, condition)
    detail = detail.copy()
    detail['battery_id'] = b
    detail['battery'] = str(battery_names[b])
    detail['seed'] = seed
    detail['condition'] = condition
    record = {'protocol_signature': signature, 'battery_id': b, 'battery': str(battery_names[b]),
              'seed': seed, 'condition': condition, 'variant': CONDITION_SPEC[condition][0],
              'n_states': len(detail), **metrics, **extra}
    if reused is not None:
        assert sha256_file(reused['states_path']) == reused['states_sha256'], 'E9 states changed after verification.'
        assert sha256_file(reused['model_path']) == reused['model_sha256'], 'E9 checkpoint changed after verification.'
        record['origin'] = 'Verified E9 anchor; no new training'
        record['e9_provenance'] = {k: reused[k] for k in ['record_sha256', 'states_sha256', 'model_sha256']}
        record['e9_provenance']['original_condition'] = ANCHOR_MAP[condition]
    else:
        record['origin'] = 'New E10 training'
    if SAVE_MODELS:
        suffix = '.ubj' if condition.endswith('XGB') else '.pt'
        model_path = run_dir / 'models' / (key + suffix)
        temporary = model_path.with_name(model_path.stem + '.tmp' + suffix)
        if reused is not None:
            shutil.copyfile(reused['model_path'], temporary)
        elif suffix == '.ubj':
            model.save_model(temporary)
        else:
            torch.save({'state_dict': model.state_dict(), 'vocab': vocab,
                        'condition': condition, 'extra': extra}, temporary)
        os.replace(temporary, model_path)
        record['model_sha256'] = sha256_file(model_path)
    atomic_csv(detail, states_path)
    record['states_sha256'] = sha256_file(states_path)
    validate_detail(detail, record, b, seed, condition)
    atomic_json(path, record)  # Completion marker written last.


def run_e10():
    run_dir, signature = prepare_e10_directory()
    assert run_dir == RUN_DIR and signature == RUN_SIGNATURE
    total = len(FOLD_IDS) * len(SEEDS) * len(CONDITIONS)
    done, newly_trained = 0, 0
    print('Planned E10 records:', total, '| new fits:', ANCHOR_REPORT.get('new_full_budget', 300), flush=True)
    for b in FOLD_IDS:
        train = [x for x in sorted(chains) if x != b]
        assert len(train) == 9 and b not in train
        vocab = build_fold_vocab(train)
        items = {x: build_scalar_state_pack(x, vocab) for x in sorted(chains)}
        item = items[b]
        width, params, target = matched_mlp_width(vocab, item['local'].shape[-1])
        status, structure = family_masks(item)
        atomic_json(run_dir / 'diagnostics' / f'features_fold{b:02d}.json',
                    {'vocab': vocab, 'local_features': item['local_names'], 'global_features': GLOBAL_NAMES,
                     'status_features': [n for n, keep in zip(item['local_names'], status) if keep],
                     'structure_features': [n for n, keep in zip(item['local_names'], structure) if keep],
                     'mlp_width': width, 'mlp_parameters': params, 'gnn_parameters': target})
        encoder = make_scalar_encoder(vocab)
        for condition in CONDITIONS:
            variant, kind = CONDITION_SPEC[condition]
            pending = []
            for seed in SEEDS:
                if completed_e10_fit(run_dir, b, seed, condition, signature):
                    done += 1
                    print(f'[{done}/{total}] resumed {b} {seed} {condition}', flush=True)
                elif (b, seed, condition) in VERIFIED_ANCHORS:
                    imported = VERIFIED_ANCHORS[(b, seed, condition)]
                    record = imported['record']
                    extra = {k: v for k, v in record.items() if k not in {
                        'protocol_signature', 'battery_id', 'battery', 'seed', 'condition', 'n_states', *METRICS}}
                    extra['e9_training_runtime_seconds'] = extra.pop('runtime_seconds', 0)
                    commit_e10_fit(run_dir, signature, b, seed, condition,
                                  {m: record[m] for m in METRICS}, pd.read_csv(imported['states_path']),
                                  extra, reused=imported, vocab=vocab)
                    done += 1
                    print(f'[{done}/{total}] imported verified E9 {b} {seed} {condition}', flush=True)
                else:
                    pending.append(seed)
            matrix = training_xgb_matrix(items, train, encoder, variant) if kind == 'xgb' and pending else None
            for seed in pending:
                started = time.time()
                print(f'Starting E10: product {b}, seed {seed}, {condition}', flush=True)
                if kind == 'mlp':
                    model, extra = train_scalar_mlp(items, train, vocab, seed, variant, epochs=EPOCHS)
                else:
                    model = make_scalar_xgb(seed)
                    model.fit(matrix[0], matrix[1], qid=matrix[2], verbose=False)
                    extra = {'training_rows': int(matrix[0].shape[0]), 'feature_count': int(matrix[0].shape[1])}
                metrics, detail = evaluate_scalar_or_gnn(model, item, kind, variant, encoder)
                extra.update(runtime_seconds=time.time() - started, device=str(DEVICE))
                commit_e10_fit(run_dir, signature, b, seed, condition, metrics, detail, extra, model=model, vocab=vocab)
                done += 1
                newly_trained += 1
                print(f'[{done}/{total}] saved {condition}; Hits@1={metrics["hits1"]:.4f}', flush=True)
                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                if MAX_NEW_FITS_THIS_CALL is not None and newly_trained >= MAX_NEW_FITS_THIS_CALL:
                    print('Per-call fit limit reached. Rerun this cell to continue; completed fits remain saved.')
                    return run_dir
            del matrix
        del items
        gc.collect()
    print('All E10 fits are complete.', flush=True)
    return run_dir


def canonical_e10_design():
    return (SEEDS == [17, 29, 43] and FOLD_IDS == sorted(chains) and len(FOLD_IDS) == 10
            and EPOCHS == 30 and XGB_TREES == 250 and NEGATIVES_PER_STATE == 8
            and LR == 2e-3 and WEIGHT_DECAY == 1e-4 and DROPOUT == 0.1
            and BASE_SEED == 20261003 and (EMBED_DIM, HIDDEN_DIM, GNN_DIM) == (12, 64, 32)
            and CONDITIONS == [v + k for k in LEARNERS for v in VARIANTS])

In [ ]:
E9_SOURCE = resolve_e9_source() if REUSE_E9_ANCHORS else None
E9_SOURCE_MANIFEST = source_manifest(E9_SOURCE)
RUN_DIR, RUN_SIGNATURE = prepare_e10_directory()
print('Protocol locked before E10 preflight/training:', RUN_DIR/'protocol.json')
print('E9 source:', E9_SOURCE if E9_SOURCE is not None else 'None; fresh anchors required')

## Cell 10 — Required preflight and E9 anchor verification
Check every state/operation combination in all ten products for the probe fold.
Verify frozen/updated families, equal dimensions, static/full endpoint equivalence,
candidate reordering, E6 state consistency, and reduced fits of all five variants.
A one-epoch MLP equivalence check compares identical initialization, training losses,
weights, and metrics for boolean E9 endpoints versus the named E10 endpoints.

Short fits are diagnostics only. Saved E9 checkpoints are then re-evaluated for all
120 eligible fits. Exact per-state metrics must match. The resulting fit budget is
printed before you launch the full experiment. No held-out score selects a variant.

In [ ]:
probe_battery = FOLD_IDS[0]
probe_train = [b for b in FOLD_IDS if b != probe_battery]
probe_vocab = build_fold_vocab(probe_train)
probe_items = {b: build_scalar_state_pack(b, probe_vocab) for b in FOLD_IDS}
STATE_DIAGNOSTICS = validate_scalar_states(probe_items, probe_vocab)
FAMILY_DIAGNOSTICS = pd.DataFrame([check_feature_families(item, probe_vocab) for item in probe_items.values()])
assert_candidate_and_metric_invariants(probe_items[probe_battery], probe_vocab)
width, params, gnn_params = matched_mlp_width(probe_vocab, probe_items[probe_battery]['local'].shape[-1])
print('Same MLP width for all variants:', width, '| parameters:', params, '| GNN target:', gnn_params)
print('Feature checks cover all ten products and every state/operation combination.')
display(FAMILY_DIAGNOSTICS)
encoder = make_scalar_encoder(probe_vocab)
smoke_rows = []
for variant in VARIANTS:
    model, info = train_scalar_mlp(probe_items, probe_train, probe_vocab, SEEDS[0], variant, epochs=1)
    metrics, detail = evaluate_scalar_or_gnn(model, probe_items[probe_battery], 'mlp', variant)
    reordered, _ = evaluate_scalar_or_gnn(model, probe_items[probe_battery], 'mlp', variant, candidate_seed=89)
    np.testing.assert_array_equal([metrics[m] for m in METRICS], [reordered[m] for m in METRICS])
    if variant in {'Static', 'Full'}:
        legacy, legacy_info = train_scalar_mlp(probe_items, probe_train, probe_vocab, SEEDS[0], variant == 'Full', epochs=1)
        assert info == legacy_info
        for name, value in model.state_dict().items():
            assert torch.equal(value, legacy.state_dict()[name]), 'Endpoint training changed: ' + name
        legacy_metrics, _ = evaluate_scalar_or_gnn(legacy, probe_items[probe_battery], 'mlp', variant == 'Full')
        np.testing.assert_array_equal([metrics[m] for m in METRICS], [legacy_metrics[m] for m in METRICS])
        del legacy
    smoke_rows.append({'condition': variant+'MLP', 'diagnostic_only': True, **metrics})
    del model
    xx, yy, qq = training_xgb_matrix(probe_items, probe_train, encoder, variant)
    tree = make_scalar_xgb(SEEDS[0], n_estimators=5)
    tree.fit(xx, yy, qid=qq, verbose=False)
    metrics, _ = evaluate_scalar_or_gnn(tree, probe_items[probe_battery], 'xgb', variant, encoder)
    reordered, _ = evaluate_scalar_or_gnn(tree, probe_items[probe_battery], 'xgb', variant, encoder, candidate_seed=89)
    np.testing.assert_array_equal([metrics[m] for m in METRICS], [reordered[m] for m in METRICS])
    smoke_rows.append({'condition': variant+'XGB', 'diagnostic_only': True, **metrics})
    del tree, xx, yy, qq
atomic_csv(STATE_DIAGNOSTICS, RUN_DIR/'diagnostics'/'preflight_e6_state_match.csv')
atomic_csv(FAMILY_DIAGNOSTICS, RUN_DIR/'diagnostics'/'preflight_feature_families.csv')
atomic_csv(pd.DataFrame(smoke_rows), RUN_DIR/'diagnostics'/'smoke_scores_not_study_results.csv')
VERIFIED_ANCHORS, ANCHOR_REPORT = verify_e9_anchors(E9_SOURCE, RUN_DIR)
display(pd.DataFrame(smoke_rows))
print('Anchor verification:', ANCHOR_REPORT)
del probe_items
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
PREFLIGHT_PASSED = True
print('Preflight passed. E10 study results have not been generated by this cell.')

## Cell 11 — Actual full E10 experiment
Set the flag below to `True` when ready to run the full locked design.
A completed E10 fit is skipped only after its protocol, state file, and model hashes
pass validation. Imported E9 anchors retain explicit provenance. A per-call fit limit
can pause the run without discarding completed work; keep running until 300 records
are complete. Resume with the same settings and E9 source.

In [ ]:
RUN_FULL_EXPERIMENT = False  # Change to True to launch the actual full study.
if not globals().get('PREFLIGHT_PASSED', False):
    raise RuntimeError('Complete Cell 10 before starting E10.')
if RUN_FULL_EXPERIMENT:
    RUN_DIR = run_e10()
else:
    print('Full run is disabled. No full-budget E10 fits were launched.')
    print('Ready budget:', ANCHOR_REPORT.get('new_full_budget', 300), 'new fits.')
    print('Set RUN_FULL_EXPERIMENT=True above and rerun this cell when ready.')

## Cell 12 — Product-level summaries and six planned primary comparisons
Average seeds within product, then weight products equally. The three contrasts
Full minus GlobalOnly, Full minus GlobalStatus, and Full minus GlobalStructure
are tested separately for each learner, in **one six-test Holm family for Hits@1**.
Exact two-sided sign-flip tests require the complete canonical design. Products
are the comparison unit; overlapping LOPO training sets limit population inference.
Bootstrap intervals are descriptive. Hits@3/MRR, stages, and singleton sensitivity
are supporting analyses. A nonsignificant difference does not establish equivalence.

In [ ]:
def exact_sign_flip(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())
    means = [abs(np.mean(values*np.array(signs))) for signs in itertools.product([-1, 1], repeat=len(values))]
    return float(np.mean(np.asarray(means) >= observed-1e-12))

def holm(pvalues):
    values = np.asarray(pvalues, dtype=float)
    order = np.argsort(values)
    result = np.empty(len(values))
    last = 0.0
    for rank, idx in enumerate(order):
        last = max(last, min(1.0, (len(values)-rank)*values[idx]))
        result[idx] = last
    return result

def summarize_e10(run_dir):
    run_dir = Path(run_dir)
    records = []
    for path in sorted((run_dir/"fits").glob("*.json")):
        rec = json.loads(path.read_text())
        assert rec['condition'] in CONDITIONS and rec['battery_id'] in FOLD_IDS and rec['seed'] in SEEDS
        assert completed_e10_fit(run_dir, rec['battery_id'], rec['seed'], rec['condition'], RUN_SIGNATURE)
        records.append(rec)
    if not records:
        raise RuntimeError("No finished fits. Run Cell 10 first.")
    all_results = pd.DataFrame([{k:v for k,v in rec.items() if k != "loss_curve"} for rec in records])
    assert not all_results.duplicated(["battery_id", "seed", "condition"]).any()
    expected_seeds = set(SEEDS)
    complete = []
    for (b, cond), group in all_results.groupby(["battery_id", "condition"]):
        if set(group.seed) == expected_seeds:
            complete.append({"battery_id": b, "battery": group.battery.iloc[0], "condition": cond,
                             **{m: float(group[m].mean()) for m in METRICS},
                             **{m+"_seed_sd": float(group[m].std(ddof=1)) for m in METRICS}})
    fold = pd.DataFrame(complete, columns=['battery_id', 'battery', 'condition', *METRICS, *[m+'_seed_sd' for m in METRICS]])
    atomic_csv(all_results, run_dir/"e10_fit_results.csv")
    atomic_csv(fold, run_dir/"e10_fold_mean.csv")
    aggregate = fold.groupby("condition", as_index=False).agg(
        hits1=("hits1", "mean"), hits3=("hits3", "mean"), mrr=("mrr", "mean"),
        n_products=("battery_id", "nunique"))
    atomic_csv(aggregate, run_dir/"e10_aggregate.csv")
    full_products = set(sorted(chains))
    pairs = PRIMARY_PAIRS
    contrasts = []
    for a, b in pairs:
        da = fold[fold.condition == a].set_index("battery_id")
        db = fold[fold.condition == b].set_index("battery_id")
        shared = sorted(set(da.index) & set(db.index))
        ready = (set(shared) == full_products and set(SEEDS) == {17,29,43}
                 and canonical_e10_design())
        for metric in METRICS:
            difference = (da.loc[shared, metric]-db.loc[shared, metric]).to_numpy()
            if len(difference):
                rng = np.random.default_rng(BASE_SEED)
                boot = rng.choice(difference, (10000, len(difference)), replace=True).mean(axis=1)
                lower, upper = np.quantile(boot, [0.025, 0.975])
                mean, wins = float(difference.mean()), int((difference>0).sum())
            else:
                lower = upper = mean = np.nan
                wins = 0
            contrasts.append({"contrast": a+" - "+b, "metric": metric,
                               "n_products": len(shared), "mean_difference": mean,
                               "bootstrap95_low": lower, "bootstrap95_high": upper,
                               "wins": wins, "full_ten_product_design": ready,
                               "p_exact": exact_sign_flip(difference) if ready else np.nan})
    contrasts = pd.DataFrame(contrasts)
    primary = contrasts.metric == "hits1"
    # Six prespecified contrasts; incomplete comparisons count as p=1.
    contrasts["p_holm"] = np.nan
    adjusted = holm(contrasts.loc[primary, "p_exact"].fillna(1.0))
    contrasts.loc[primary, "p_holm"] = np.where(contrasts.loc[primary, "p_exact"].notna(), adjusted, np.nan)
    atomic_csv(contrasts, run_dir/"e10_contrasts.csv")
    state_frames = [pd.read_csv(path) for path in sorted((run_dir/"fits").glob("*_states.csv.gz"))]
    details = pd.concat(state_frames, ignore_index=True)
    details["stage"] = pd.cut(details.progress, [-1, 1/3, 2/3, 1.001],
                              labels=["early", "middle", "late"], right=False)
    stages = details.groupby(["battery_id", "condition", "seed", "stage"], observed=True)[METRICS].mean().reset_index()
    stages = stages.groupby(["battery_id", "condition", "stage"], observed=True)[METRICS].mean().reset_index()
    atomic_csv(stages, run_dir/"e10_stage_product_metrics.csv")
    nonsingleton = details[details.candidate_size > 1]
    singleton = nonsingleton.groupby(["battery_id", "condition", "seed"])[METRICS].mean().reset_index()
    singleton = singleton.groupby(["battery_id", "condition"])[METRICS].mean().reset_index()
    atomic_csv(singleton, run_dir/"e10_without_singleton_fold.csv")
    expected = len(FOLD_IDS)*len(SEEDS)*len(CONDITIONS)
    status = {"requested_fits": expected, "finished_fits": len(all_results),
              "full_canonical_design": canonical_e10_design(),
              "all_requested_fits_complete": len(all_results) == expected,
              "verified_e9_anchors": int((all_results.origin == "Verified E9 anchor; no new training").sum()),
              "new_training_records": int((all_results.origin == "New E10 training").sum())}
    atomic_json(run_dir/"e10_completion.json", status)
    display(aggregate.sort_values("hits1", ascending=False))
    display(contrasts[primary])
    print("Completion:", status)
    if not status["full_canonical_design"]:
        print("DIAGNOSTIC RUN: these outputs are not the full manuscript experiment.")
    return aggregate, fold, contrasts, stages

In [ ]:
if list((RUN_DIR/'fits').glob('*.json')):
    AGGREGATE, PRODUCT_RESULTS, CONTRASTS, STAGE_RESULTS = summarize_e10(RUN_DIR)
    if not AGGREGATE.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        plot = AGGREGATE.sort_values('hits1')
        ax.barh(plot.condition, plot.hits1)
        ax.set_xlabel('Mean product-level Hits@1')
        ax.set_title('E10 feature-family experiment — check completion before inference')
        fig.tight_layout()
        fig.savefig(RUN_DIR/'e10_hits1.png', dpi=180)
        plt.show()
else:
    atomic_json(RUN_DIR/'e10_completion.json', {'requested_fits': 300, 'finished_fits': 0,
                'all_requested_fits_complete': False, 'full_canonical_design': canonical_e10_design(),
                'status': 'Preflight only; no full-budget study results'})
    print('No E10 study fits yet. Diagnostics are not empirical E10 results.')

## Cell 13 — Export and interpretation
The ZIP contains the locked protocol, provenance, saved fits/models, feature checks,
and summaries when available. It excludes the raw dataset. A preflight-only ZIP
is labelled incomplete and must not be used as study evidence.

If Full exceeds GlobalOnly, local updates jointly add predictive value beyond the
global fractions. Full minus GlobalStatus tests additional local structure updates;
Full minus GlobalStructure tests additional local status updates. These effects are
conditional on this frozen-family design and may include interactions.

Interpret all planned outcomes, including unfavorable ones. After E10, execute the
separately specified E11 degree/activity-controlled adjacency audit. Neither E10 nor
the notebook's literature notes establish Q1 readiness or a first-study claim.

In [ ]:
readme = """E10 state-feature-family follow-up
Read e10_completion.json before interpreting any result. Reduced smoke fits in
diagnostics are NOT study results. The design was selected after E9 and locked before
E10 training. There are 300 fit records in the complete design. All 120 E9 endpoints
must pass source/protocol/version/checkpoint/per-state numerical checks before any
anchor is reused; otherwise all anchors are retrained. Imported model payloads retain
their original E9 condition and provenance; E10 fit records identify the new variant.
Unused feature families retain initial-state values; all columns and capacities stay
constant. Sequence progress, static operation features, initial degrees, and shared
component features are common. State updating follows the original simplified E6 rule.
Six planned primary Hits@1 contrasts share one Holm family. Seeds are averaged within
product. Products are weighted equally. Exact sign-flip inference assumes product-level
sign exchangeability; overlapping LOPO training sets and ten products limit generality.
Intervals, stages, and singleton analyses are descriptive. No nonsignificance/equivalence
or physical-causality claim is supported. E11 is separate and has not been run here.
Raw data: DOI 10.35097/emz24pksshndq468. The ZIP contains no raw KIT data.
"""
(RUN_DIR/'README.txt').write_text(readme, encoding='utf-8')
ZIP_PATH = OUTPUT_ROOT/'e10_state_feature_families_results.zip'
temporary_zip = ZIP_PATH.with_suffix('.tmp.zip')
with zipfile.ZipFile(temporary_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file() and '.tmp' not in path.name:
            z.write(path, str(path.relative_to(RUN_DIR)))
os.replace(temporary_zip, ZIP_PATH)
print('Result ZIP:', ZIP_PATH)
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))